# Week 7 In-Class Exercise: Ensemble Learning and Random Forests

In the textbook (Chapter 7), we saw that combining many models often beats any single one. A **voting classifier** polls several different models; **bagging** trains the *same* kind of model on many bootstrap resamples; a **Random Forest** is bagging applied to decision trees with an extra dose of feature randomness; and **boosting** (AdaBoost, Gradient Boosting) builds trees in sequence, each one fixing the previous one's mistakes. Those demos mostly used the two-moons toy set, iris, and MNIST.

In this exercise you'll apply those **same ideas** to a dataset that is new to this course and a lot messier than a toy:

> **German Credit ("credit-g")** — 1,000 loan applicants, each described by 20 features that mix **numbers** (loan duration, credit amount, age, ...) with **categories** (checking-account status, credit history, employment, housing, ...). The label is whether the applicant is a **good** or **bad** credit risk. The classes are **imbalanced** (700 good, 300 bad) and the signal is weak, so a single decision tree struggles — which is exactly the situation where ensembles earn their keep.

The dataset is downloaded once from OpenML (`fetch_openml(name='credit-g')`) and then cached. Because the features are a mix of numeric and categorical, we hand you a small preprocessing pipeline in Step 1 so you can focus on the ensemble methods, not the data wrangling.

Every technique from the textbook transfers directly — adapt the code, don't invent new methods.

1. Load and explore the data (provided)
2. A **single Decision Tree** baseline
3. A **Voting Classifier** (hard vs soft)
4. **Bagging** many trees
5. **Out-of-bag** evaluation (a free validation set)
6. A **Random Forest** and its **feature importances**
7. **Boosting** — watch it overfit as trees pile up
8. Pick a winner on the **test set**

### Useful API References

| Task | Documentation |
|------|---------------|
| Load data | [fetch_openml](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html) |
| Train/test split | [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) |
| Decision tree | [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) |
| Voting | [VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) |
| Bagging | [BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html) |
| Random forest | [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) |
| Gradient boosting | [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) |
| Cross-validation | [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) |
| Metrics | [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) |

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/pjmcswee/IST707-Notebooks/blob/main/week7/week7_inclass_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

## Setup

Run this cell to import the libraries and set the plotting defaults.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rc('font', size=13)
plt.rc('axes', labelsize=13, titlesize=13)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## Step 1: Load the Data (Provided)

The textbook Chapter 7 demos used the two-moons toy set and iris. Here we load the **German Credit** dataset from OpenML. Because its 20 features mix numbers and categories, this cell also builds a `preprocess` transformer (scale the numbers, one-hot encode the categories) and a stratified 75/25 train/test split. This cell is provided — just run it.

Everywhere below, wrap a model in a pipeline like `make_pipeline(preprocess, model)` so it receives numeric input. The target is encoded as `1 = good`, `0 = bad`.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

credit = fetch_openml(name='credit-g', version=1, as_frame=True, parser='auto')
df = credit.frame.copy()

# Target: 1 = good credit risk, 0 = bad credit risk
y = (df['class'] == 'good').astype(int).values
X = df.drop(columns='class')

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

# Reusable preprocessing: scale numbers, one-hot encode categories.
preprocess = make_column_transformer(
    (StandardScaler(), num_cols),
    (OneHotEncoder(handle_unknown='ignore'), cat_cols),
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

print('Total samples:', X.shape[0], ' Features:', X.shape[1])
print(f'Numeric features ({len(num_cols)}):', num_cols)
print(f'Categorical features ({len(cat_cols)}):', cat_cols)
print('Class balance (full set): good =', int(y.sum()), ' bad =', int((1 - y).sum()))
print('Train:', X_train.shape[0], ' Test:', X_test.shape[0])
print('\nMajority-class baseline (always guess "good"):',
      round(max(y_train.mean(), 1 - y_train.mean()), 3))
X.head()

## Step 2: A Single Decision Tree (the baseline to beat)

Everything in Chapter 7 is about combining models, so first we need a single model to compare against. A lone decision tree is a natural choice because every ensemble in this notebook is built out of trees.

**Your turn!** Build a `make_pipeline(preprocess, DecisionTreeClassifier(random_state=42))`, fit it on the training data, and print the train and test accuracy. Note how far the train accuracy runs ahead of the test accuracy.

**Hint (from the textbook):**
```python
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

tree = make_pipeline(preprocess, DecisionTreeClassifier(random_state=42))
tree.fit(X_train, y_train)
accuracy_score(y_test, tree.predict(X_test))
```

In [ ]:
# TODO: Fit a single DecisionTreeClassifier (inside make_pipeline(preprocess, ...)).
#       Print the train accuracy and the test accuracy.
# Your code here:


**Question:** The train accuracy is near 1.0 while the test accuracy is far lower. What is the single tree doing to earn that near-perfect training score, and why should the *test* number — not the train number — be the one you compare every ensemble against?

*Your answer:*


## Step 3: A Voting Classifier

The textbook's first ensemble polls **several different kinds of model** and takes the majority vote. **Hard** voting counts predicted labels; **soft** voting averages predicted probabilities and usually does a little better when every model can output calibrated probabilities.

**Your turn!** Build a `VotingClassifier` over three different models — `LogisticRegression`, `RandomForestClassifier`, and `SVC` — each wrapped so it sees preprocessed data. Fit it with `voting='hard'` and then `voting='soft'` (remember `SVC(probability=True)` for soft voting), and print the test accuracy of each.

**Hint (from the textbook):**
```python
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

voting = VotingClassifier(
    estimators=[
        ('lr', make_pipeline(preprocess, LogisticRegression(random_state=42))),
        ('rf', make_pipeline(preprocess, RandomForestClassifier(random_state=42))),
        ('svc', make_pipeline(preprocess, SVC(probability=True, random_state=42))),
    ],
    voting='hard')
```

In [ ]:
# TODO: Build a VotingClassifier over LogisticRegression, RandomForestClassifier,
#       and SVC (each inside make_pipeline(preprocess, ...)).
#       Fit and print test accuracy for voting='hard' and voting='soft'.
# Your code here:


**Question:** How does the voting classifier's test accuracy compare to the single tree from Step 2? The textbook says a voting ensemble works best when its members make **independent** errors. Why does mixing three *different kinds* of model (a linear model, a forest, an SVM) help satisfy that condition better than voting among three decision trees would?

*Your answer:*


## Step 4: Bagging

Bagging (**b**ootstrap **agg**regat**ing**) takes the *opposite* approach to Step 3: instead of several different models, it trains **many copies of the same model** — here, decision trees — each on a different bootstrap resample of the training data, then averages their votes. The individual trees still overfit, but their errors partly cancel.

**Your turn!** Build a `BaggingClassifier` of 300 `DecisionTreeClassifier` estimators (inside the preprocessing pipeline), fit it, and print its test accuracy next to the single-tree test accuracy from Step 2.

**Hint (from the textbook):**
```python
from sklearn.ensemble import BaggingClassifier
bag = make_pipeline(
    preprocess,
    BaggingClassifier(DecisionTreeClassifier(random_state=42),
                      n_estimators=300, n_jobs=-1, random_state=42))
```

In [ ]:
# TODO: Build a BaggingClassifier of 300 decision trees (inside make_pipeline).
#       Fit it and print its test accuracy next to the single-tree accuracy.
# Your code here:


**Question:** The individual trees inside the bag are the *same* high-variance trees you saw overfit in Step 2, yet the bagged ensemble generalizes better. In one or two sentences, explain how averaging many overfit trees reduces **variance** without raising **bias**.

*Your answer:*


## Step 5: Out-of-Bag Evaluation

Each bootstrap sample leaves out roughly a third of the training rows. Those left-out ("out-of-bag") rows are a ready-made validation set: every tree can be scored on the rows it never saw, for free, without touching the test set.

**Your turn!** Rebuild the bagging classifier with `oob_score=True`. Fit it on the training data and print the **OOB score**. Then compare that OOB estimate to the actual **test accuracy** of the same model.

Because `oob_score_` lives on the `BaggingClassifier` (not the outer pipeline), it is easiest here to transform the data once and fit the bag on the result:

**Hint:**
```python
Xtr = preprocess.fit_transform(X_train)
Xte = preprocess.transform(X_test)
bag = BaggingClassifier(DecisionTreeClassifier(random_state=42),
                        n_estimators=300, oob_score=True,
                        n_jobs=-1, random_state=42).fit(Xtr, y_train)
bag.oob_score_                       # out-of-bag accuracy estimate
accuracy_score(y_test, bag.predict(Xte))
```

In [ ]:
# TODO: Fit a BaggingClassifier(oob_score=True) on preprocessed training data.
#       Print the OOB score and the test accuracy, and compare them.
# Your code here:


**Question:** How close is the OOB score to the test accuracy? Explain *why* the OOB rows give a fair estimate of generalization even though they came from the training set — and name one practical advantage of OOB scoring over setting aside a separate validation split.

*Your answer:*


## Step 6: Random Forest and Feature Importances

A Random Forest is bagging specialized to trees, with one extra twist: at each split it considers only a **random subset of the features**. That decorrelates the trees even further. A bonus is that the trained forest can rank the features by how much each one reduced impurity across all its trees.

**Your turn (part A)!** Fit a `RandomForestClassifier(n_estimators=300, random_state=42)` (inside the pipeline) and print its test accuracy.

**Your turn (part B)!** Pull the `feature_importances_` out of the fitted forest and plot the **top 10** features as a horizontal bar chart. The feature names after one-hot encoding come from `preprocess.get_feature_names_out()`.

**Hint:**
```python
rf_pipe = make_pipeline(preprocess,
                        RandomForestClassifier(n_estimators=300, random_state=42))
rf_pipe.fit(X_train, y_train)
forest = rf_pipe.named_steps['randomforestclassifier']
names = rf_pipe.named_steps['columntransformer'].get_feature_names_out()
importances = forest.feature_importances_   # one value per (expanded) feature
```

In [ ]:
# TODO (A): Fit a RandomForestClassifier(n_estimators=300) and print test accuracy.
# TODO (B): Plot the top 10 features by feature_importances_ as a horizontal bar chart.
#           (Feature names come from the fitted preprocess step's get_feature_names_out().)
# Your code here:


**Question:** Which features does the forest rely on most to judge credit risk? Do the top-ranked features match your common-sense intuition about who repays a loan? Name one way a loan officer could use this importance ranking.

*Your answer:*


## Step 7: Boosting — Watch Gradient Boosting Overfit

Bagging and forests build their trees **in parallel** and average them. Boosting builds trees **in sequence**, each new tree focusing on the examples the previous ones got wrong (AdaBoost re-weights the misclassified samples; Gradient Boosting fits each new tree to the ensemble's residual errors). The textbook makes a sharp claim: because boosting keeps chasing the training set, **adding too many trees makes it overfit** — something bagging and forests essentially never do. Let's see that happen.

**Your turn!** Sweep the number of boosting rounds. For `n_estimators` in `[5, 25, 100, 500, 2000]`, fit a `GradientBoostingClassifier(learning_rate=0.1, max_depth=2, random_state=42)` (inside the pipeline), and record both **train** and **test** accuracy. Then plot both curves against `n_estimators` (use a log scale on the x-axis). Watch the train curve climb toward 1.0 while the test curve peaks early and then sags.

**Hint (from the textbook):**
```python
from sklearn.ensemble import GradientBoostingClassifier
n_list = [5, 25, 100, 500, 2000]
for n in n_list:
    gb = make_pipeline(preprocess,
        GradientBoostingClassifier(n_estimators=n, learning_rate=0.1,
                                   max_depth=2, random_state=42))
    gb.fit(X_train, y_train)
    # record gb.score(X_train, y_train) and gb.score(X_test, y_test)
# plt.semilogx(n_list, train_scores, ...); plt.semilogx(n_list, test_scores, ...)
```

In [ ]:
# TODO: Sweep n_estimators over [5, 25, 100, 500, 2000] for a
#       GradientBoostingClassifier. Collect train AND test accuracy for each.
#       Plot both curves vs n_estimators on a log-scale x-axis, and print the table.
# Your code here:


**Question:** As `n_estimators` grows, the train accuracy marches toward 1.0 but the test accuracy peaks at a modest number of trees and then *stops improving or drifts down*. Roughly how many trees gave the best test accuracy, and what is the widening gap between the two curves telling you? Based on *how* boosting builds its trees (each one dependent on the last), explain why a Random Forest could keep adding trees without this happening.

*Your answer:*


## Step 8: Pick a Winner

You now have several ensembles. Time to choose one the honest way — with cross-validation on the **training** set, not by peeking at the test set.

**Your turn!** Use 5-fold `cross_val_score` on the **training** data to compare at least three models from this notebook (for example: the single tree, the random forest, and gradient boosting). Print each model's mean CV accuracy, choose the best one, and *only then* report that single winner's accuracy on the held-out test set.

**Hint:**
```python
from sklearn.model_selection import cross_val_score
models = {
    'tree': make_pipeline(preprocess, DecisionTreeClassifier(random_state=42)),
    'forest': make_pipeline(preprocess, RandomForestClassifier(n_estimators=300, random_state=42)),
    'gbrt': make_pipeline(preprocess, GradientBoostingClassifier(random_state=42)),
}
for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=5)
```

In [ ]:
# TODO: 5-fold cross_val_score on the TRAINING set for >=3 models.
#       Print each mean CV accuracy, pick the best, then report ONLY that
#       model's accuracy on the test set.
# Your code here:


**Question:** Did the model with the best cross-validation score also give the best test accuracy? Why is choosing the winner by cross-validation (and touching the test set only once, at the very end) the trustworthy way to report a final number?

*Your answer:*


## Reflection Questions

Answer each question in the cell below it (1-3 sentences each).

**Q1:** Across Steps 2–7 most ensembles beat the single tree, often only by a few points. On this noisy, imbalanced credit data, why do the gains from ensembling tend to be modest rather than dramatic — and what does that say about the limits of fancier models when the underlying signal is weak?

*Your answer:*


**Q2:** **Bagging** and **boosting** both combine many trees but in opposite ways. In one sentence each, describe how each one builds its trees and which error it mainly attacks (variance vs bias).

*Your answer:*


**Q3:** The target is imbalanced (700 good, 300 bad), and a model that always guesses "good" already scores about 0.70. Why can **accuracy** be a misleading score here, and what would a bank care about that plain accuracy hides (think about the cost of approving a loan that defaults)?

*Your answer:*


**Q4:** A Random Forest gives you a ranked list of feature importances almost for free, while a soft-voting ensemble of a logistic model, an SVM, and a forest does not. If a regulator demanded that you *explain* why a specific applicant was denied credit, which of your Step 3–7 models would be easiest to defend, and why?

*Your answer:*
